In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
 
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2025-12-02 23:38:00 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



# Hallazgos


Tabla `resultados_wompi.wompi_merchants`

- La tabla se ingesta full, pero se borran las ingestas full del pasado
- Los registros son únicos por la variable id_comercio
- Un comercio puede tener varios id_comercio ¿Cómo interpretar esto? ¿Tiene sentido? ¿Cómo gestionarlo? ya que por documento_identidad y tipo_documento hay varios registros. Ejemplo: 800253799. Respuesta: Un comercio como Frisby puede tener varios locales y cada uno un id_comercio [id wompi] diferente
- Modelo = 'Agregador' porque significan que dan datáfono, link pagos por pse y botón bancolombia. 'Gateway' acopla todos los medios de pago vinculados al pago se acopla la adquirencia entonces se estaría doble contando. Otra explicación que da *Daniel Ramirez Vergara* es Wompi Agregador es una solucion que entrega los medios de pago (adquirencia, botones, puntos y demas) Gateway es una forma de llevar los productos que el cliente tiene con bancolombia al mundo digital ! 


## Datos importantes

- Desde Junio 2025 [aunque se ve un aumento significativo en mayo 2025] se iniciaron las acciones comerciales desde el equipo comercial. [Tanto adquirencia como wompi]
- En agosto inicio nequi negocios


¿Qué tal este query para obtener el historico de vinculaciones a wompi?

In [2]:
dict_ult_ing_wompi_merch = helper.obtener_ultima_ingestion('resultados_wompi.wompi_merchants')
dict_ult_ing_wompi_merch

2025-12-02 23:38:04 - [INFO] - Buscando fechas para resultados_wompi.wompi_merchants
2025-12-02 23:38:04 - [INFO] - Transcurrido: 1764736685, Tiempo de Refresco = 1000
/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:476: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, cn)
2025-12-02 23:38:06 - [INFO] - Finalizo la busqueda, duracion: 00:01.7, resultado: {'year': 2025, 'month': 12, 'day': 2}


{'year': 2025, 'month': 12, 'day': 2}

In [10]:
sql = """
SELECT CASt(left(cast(creado as string), 6) as int) as creado_ym, count(*) as num_vinc
FROM resultados_wompi.wompi_merchants
WHERE YEAR = """ + str(dict_ult_ing_wompi_merch['year']) + """
  AND MONTH = """ + str(dict_ult_ing_wompi_merch['month']) + """
  AND DAY = """ + str(dict_ult_ing_wompi_merch['day']) + """
  AND modelo = 'Agregador'
  and activo = 'A'
  and desembolsos_permitidos = 'Si'
GROUP BY 1
ORDER BY creado_ym;
"""
helper.obtener_dataframe(sql)

------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 4/4 DATAFRAME         ejecutando   08:54:16 AM             

2025-11-13 08:54:17 - [INFO] - 86 filas, 2 columnas, 00:00.6 consultando, 00:00.0 descargando, 00:00.0 convirtiendo


 4/4 DATAFRAME         finalizado   08:54:16 AM     00:00.7 
------------------------------------------------------------


,creado_ym,num_vinc
0,201807,1
1,201810,1
2,201811,1
3,201812,1
4,201902,1
...,...,...
81,202507,4330
82,202508,7090
83,202509,14925
84,202510,12637


In [ ]:
# La ingestio